# Practical 1 - From Airfoil Geometry to Lift and Drag
## Part II: training the surrogate

You have a dataset.

Everything from here is free, you can retrain as often as you like. What you
**cannot** do is fix a dataset that missed something. If your model turns out
to be blind somewhere, that is now a fact about your afternoon, and the useful
response is to find out *where* rather than to train longer.

You need a saved surrogate that satisfies the workshop. Section 6 checks
that for you. Run it with time to spare.

In [ ]:
# Clone repo
%cd /content
! [ -d "/content/practical" ] && echo "Repository already cloned" || git clone https://github.com/ArnauMiro/BIP-Torino-practical.git practical
%cd /content/practical

# Colab setup. Skip locally if you already have these.
%pip install -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import P1_ancillary as p1

X, Y, conf, meta = p1.load_campaign("") # Change "" for your dataset
print(f"{X.shape[0]} rows, spent {meta['spent']}/{meta['budget']}")
print("campaign card:", meta["campaign_card"].get("expected_weaknesses", "(empty)"))

---
## 1 · One line

That is the whole API. `train_surrogate` wraps pyLOM's MLP so you can get a
working model immediately and spend your time on the parts of this session that
matter.

In [ ]:
model = p1.train_surrogate(X, Y)

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(model.logs["train_loss"], label="train", lw=.8)
n_ep = len(model.logs["test_loss"])
step = max(1, len(model.logs["train_loss"]) // n_ep)
plt.plot(np.arange(step, step * n_ep + 1, step), model.logs["test_loss"],
         label="validation")
plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("loss")
plt.legend(); plt.tight_layout(); plt.show()

---
## 2 · The three knobs that are actually decisions

Everything below is a keyword argument to `train_surrogate`. Two of them are
about the model. **One of them is not**, and it is the one that matters most.

```python
model = p1.train_surrogate(
    X, Y,
    target="log10",           # or "raw"
    split="airfoil",          # or "random"
    envelope_quantile=0.99,   # lower is stricter
    arch={"hidden_size": 96, "n_layers": 3, "p_dropouts": 0.0}, # MLP architecture
    training_params={"epochs": 300, "lr": 1e-3, "batch_size": 64}, # MLP training params
)
```

**`target`**: it asks for `log10(Cd)`, and the default gives you that.
Look at the spread of Cd in your dataset, then think about what an MSE loss does
to a target that spans a decade. Try `"raw"` if you want to see it rather than
believe it.

**`split`**: `"airfoil"` puts every angle of a given geometry on one side of the
train/validation line. `"random"` splits rows. Try both and compare the
validation curves. One of them will look better. Think hard about whether it
*is* better, and what your validation error is measuring in each case.

**`envelope_quantile`**: how far from your training data a query can sit before
`in_envelope` returns False. This is not a model knob. It is a claim about what
you know, and on Wednesday an agent will test that claim.

In [ ]:
# Your experiments. Retraining is free -- the data was the expensive part.
alt = p1.train_surrogate(X, Y, split="random", verbose=False)

for name, m in (("airfoil split", model), ("random split", alt)):
    print(f"{name:<16} final validation loss {m.logs['test_loss'][-1]:.5f}")

- Which looks better? 
- Which *is* better? 

You cannot answer the second question with anything in this notebook, and that is the point.

---
## 3 · Where is it wrong?

Aggregate error hides the thing you need. Look at the residual against each
input separately.

In [ ]:
tr, va = p1.split_by_airfoil(X, frac=0.2, seed=0)
Ztrue = p1.TargetTransform.forward(Y)[va]
Zpred = model.predict(X[va])

fig, ax = plt.subplots(2, 4, figsize=(14, 6))
for j, lab in enumerate(p1.COLUMNS):
    for k, out in enumerate(("Cl", "log10(Cd)")):
        ax[k, j].scatter(X[va, j], Zpred[:, k] - Ztrue[:, k], s=6, alpha=.35)
        ax[k, j].axhline(0, color="k", lw=.7)
        ax[k, j].set_xlabel(lab)
        if j == 0:
            ax[k, j].set_ylabel(f"error in {out}")
plt.tight_layout(); plt.show()

Questions worth arguing about before you move on:

* Is the error flat across α, or does it grow toward one end? Which end, and does
  that match where you spent your budget?
* Is the Cl error and the Cd error worst in the same place, or different places?
* Does the residual look like noise, or does it have structure? Structure means
  the model is missing something systematic, and no amount of extra epochs
  fixes that.

---
## 4 · The envelope

`in_envelope` is the most important method your surrogate has and probably the
one you will spend the least time on. Wednesday's agent searches your action
space for wherever your model promises the most lift for the least drag. If your
model is confidently wrong somewhere, it *will* find that place, and this
function returning False is the only defence you have.

Below: prediction error against distance to the nearest training point. If error
does not grow with distance, your envelope is decorative.

In [ ]:
d_nn = model.envelope.nn_distance(X[va])
err = np.abs(Zpred[:, 1] - Ztrue[:, 1])

plt.figure(figsize=(6.5, 4))
plt.scatter(d_nn, err, s=8, alpha=.4)
plt.axvline(model.envelope.threshold_, color="k", ls="--", lw=1,
            label=f"threshold (q={model.envelope.quantile})")
plt.xlabel("scaled distance to nearest training point")
plt.ylabel("|error| in log10(Cd)"); plt.yscale("log")
plt.legend(); plt.tight_layout(); plt.show()

probe = np.array([
    [3.0, 0.45, 12.0,  5.0],    # middle of the clamp
    [3.0, 0.45, 22.0,  5.0],    # thick, outside
    [3.0, 0.45, 12.0, 16.0],    # stalled, outside
])
print("in_envelope:", model.in_envelope(probe))
print("predict:\n", model.predict(probe).round(4))

Note that `predict` answered all three. It must, returning NaN or raising would
break Wednesday's environment. Expressing doubt is `in_envelope`'s job, not
`predict`'s.

---
## 5 · Save it

In [ ]:
GROUP = ""          # <- your group name. This is what appears on the leaderboard.
assert GROUP, "name your group"

model.save(GROUP)
print(f"saved to ./{GROUP}/")

---
## 6 · Check

Run this before you submit. It reloads your surrogate from disk, exactly as
Practical 3 will on Wednesday, and checks the four things that would otherwise
fail in front of the room.

In [ ]:
reloaded = p1.PyLOMSurrogate.load(GROUP)
probe = np.array([[3.0, 0.45, 12.0, 5.0], [1.0, 0.30, 9.0, 2.0]])
ok = True

Z = reloaded.predict(probe)
if Z.shape != (2, 2):
    print(f"FAIL  predict returned {Z.shape}, contract is (N, 2)"); ok = False

wild = np.array([[9.0, 0.85, 30.0, 25.0], [0.0, 0.20, 4.0, -5.0]])
if not np.all(np.isfinite(reloaded.predict(wild))):
    print("FAIL  predict returned non-finite values far outside the "
          "envelope. It must always answer; use in_envelope to doubt."); ok = False

e = reloaded.in_envelope(probe)
if e.shape != (2,) or e.dtype != bool:
    print(f"FAIL  in_envelope returned {e.shape} {e.dtype}, want (N,) bool")
    ok = False

if not np.allclose(Z, model.predict(probe), atol=1e-5):
    print("FAIL  reloaded model disagrees with the one in memory"); ok = False

print("\nOK -- ready for Wednesday" if ok else "\nfix the above")

---

Submit the folder. Keep your campaign card open.

You will be asked to explain the gap between what you predicted your
campaign would be good at and what the three leaderboards actually say. The
groups that predicted their own weakness correctly get more credit than the
groups that came first.